# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. Unit of Analysis & Time Window

* **Unit of Analysis (Grain):** One row represents one unique content item per client (`client_hash_id` $\times$ `content_hash_id`), aggregated across the 30-day feature window.
* **Time Window:** A 30-day historical window covering **March 1, 2026 to March 31, 2026** (`report_date >= '2026-03-01' AND report_date <= '2026-03-31'`).
* **Output:** A verified feature dataset of 331,437 rows where each row represents a unique content item's 30-day aggregated performance with classified features, target label (`total_impressions`), missingness indicators, and context for downstream model training.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
try:
    if not HF_TOKEN:
        HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
except Exception:
    HF_TOKEN = ''

In [3]:
import duckdb

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Verify uniqueness at the stated grain (client_hash_id x content_hash_id for March 2026)
grain_check_query = """
WITH march_aggregated AS (
    SELECT 
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
        COUNT(DISTINCT report_date) AS active_days
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
)
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    COUNT(*) - COUNT(DISTINCT content_hash_id) AS duplicate_check
FROM march_aggregated;
"""

grain_results = con.execute(grain_check_query).df()
print(grain_results)

   total_rows  unique_content_items  unique_clients  duplicate_check
0      331437                331437              55                0


Verified that `duplicate_check` is 0 across all 331,437 aggregated rows, confirming that the stated grain (`client_hash_id` $\times$ `content_hash_id`) holds strictly without duplicate items.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Label (Target Metric)
* `total_impressions`: Sum of `gsc_impressions` aggregated over the 30-day window (`March 1–31, 2026`). Represents total search engine visibility.

### Features (Knowable BEFORE prediction window)
* **Search Performance & Rank Signals:**
  * `total_clicks`: Sum of `gsc_clicks` aggregated over the 30-day window.
  * `avg_position`: Weighted average rank position derived from `SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0)`.
  * `active_days`: Count of distinct `report_date` instances with active measurement (`COUNT(DISTINCT report_date)`).
  * `historical_ctr`: Derived click-through rate calculated as `SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)`.
* **User Engagement & Analytics Signals (GA4):**
  * `total_pageviews`: Sum of `ga4_pageviews` over the 30-day window.
  * `total_sessions`: Sum of `ga4_sessions` over the 30-day window.
  * `engagement_rate`: Derived metric from GA4 computed as `SUM(ga4_engaged_sessions) / NULLIF(SUM(ga4_sessions), 0)`.
  * `avg_engagement_time`: Engagement duration per user calculated as `SUM(ga4_total_engagement_sec) / NULLIF(SUM(ga4_users), 0)`.
  * `organic_session_ratio`: Proportion of traffic from organic search computed as `SUM(sessions_organic) / NULLIF(SUM(ga4_sessions), 0)`.
  * `ai_referral_sessions`: Combined traffic from AI search engines (`SUM(ai_chatgpt + ai_perplexity + ai_gemini + ai_copilot + ai_claude + ai_meta + ai_other)`).

### Context (For grouping, joining, splitting — never learned by model)
* `client_hash_id`: Anonymized unique identifier for each client. Used for client-grouped cross-validation splits.
* `content_hash_id`: Anonymized unique identifier for each content piece.
* `client_has_gsc`: Boolean indicator verifying whether Google Search Console tracking is enabled for the client.
* `client_has_ga4`: Boolean indicator verifying whether GA4 analytics tracking is enabled for the client.

### Excluded (With clear justification)
* `report_date`: Excluded because raw daily timestamps violate the aggregated 30-day unit of analysis (`client_hash_id` $\times$ `content_hash_id`).
* `gsc_sum_position`: Excluded as a raw feature because it is an unnormalized running sum; used only internally to derive the normalized `avg_position`.
* Individual daily referral columns (`ai_chatgpt`, `ai_perplexity`, etc.): Excluded in raw form to reduce feature sparsity; aggregated into `ai_referral_sessions`.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 1. Grain Verification Query
Proves zero duplicates at the `client_hash_id` $\times$ `content_hash_id` grain for the 30-day window.

In [4]:
print(grain_results)

   total_rows  unique_content_items  unique_clients  duplicate_check
0      331437                331437              55                0


### 2. Search Signals, GA4 Signals & Missingness Verification Query
Aggregates raw daily rows to the defined grain, safely handles division by zero using `NULLIF(..., 0)`, measures exact missingness rates, and computes summary statistics.

In [5]:
claim_check_query = """
WITH feature_summary AS (
    SELECT 
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
        COUNT(DISTINCT report_date) AS active_days,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS historical_ctr,
        SUM(ga4_pageviews) AS total_pageviews,
        SUM(ga4_sessions) AS total_sessions,
        SUM(ga4_engaged_sessions) / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
)
SELECT 
    COUNT(*) AS total_rows,
    -- Null percentages
    AVG(CASE WHEN avg_position IS NULL THEN 1.0 ELSE 0.0 END) AS null_avg_position_pct,
    AVG(CASE WHEN engagement_rate IS NULL THEN 1.0 ELSE 0.0 END) AS null_engagement_rate_pct,
    -- Summary Statistics
    AVG(total_impressions) AS mean_impressions,
    AVG(avg_position) AS mean_position,
    AVG(engagement_rate) AS mean_engagement_rate,
    MAX(active_days) AS max_active_days
FROM feature_summary;
"""
print(con.execute(claim_check_query).df())

   total_rows  null_avg_position_pct  null_engagement_rate_pct  mean_impressions  mean_position  mean_engagement_rate  max_active_days
0      331437               0.466752                  0.727740        846.790156       15.99227              0.025977               31


### 3. Client & Tracking Integration Query
Verifies total unique clients, unique content items, and client tracking coverage flags (`client_has_gsc`, `client_has_ga4`).

In [6]:
content_check_query = """
SELECT 
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    COUNT(DISTINCT content_hash_id) AS unique_contents,
    COUNT(CASE WHEN client_has_gsc = TRUE THEN 1 END) AS rows_with_gsc,
    COUNT(CASE WHEN client_has_ga4 = TRUE THEN 1 END) AS rows_with_ga4
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31';
"""
print(con.execute(content_check_query).df())

   unique_clients  unique_contents  rows_with_gsc  rows_with_ga4
0              55           331437        9841378        6822637


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Key Data Limits & Traps Identified

1. **Unbalanced Client History & Depth:**
   * Client onboarding dates (`gsc_data_start`) vary across the dataset. Historical depth is non-uniform, meaning per-client performance comparisons must use relative per-client time windows rather than assuming a single fixed global historical baseline.

2. **Structural GA4 Non-Tracking & Zero-Filling (MNAR):**
   * **30.69%** of daily rows in March 2026 have `ga4_data_available = FALSE`. Rows prior to a client's GA4 integration are zero-filled rather than recorded as missing.
   * A naive `fillna(0)` treats non-tracked clients as having zero engagement, creating severe category distortion. Models must use explicit `ga4_data_available` boolean context flags or drop non-tracked client rows.

3. **Null Position for Zero-Impression Items:**
   * **46.68%** of aggregated content rows have `null_avg_position` because `gsc_impressions = 0`. In Google Search Console data, position = 0 indicates "no rank data available", not rank #0. Position metrics must strictly use `NULLIF(impressions, 0)`.

4. **Window Alignment & Target Leakage Risks:**
   * Search query tables (`fact_content_query_90d`) cover a 90-day window. Features derived from query metrics that overlap with the label prediction window introduce severe future leakage. Only features strictly computed prior to the target prediction start date (`< 2026-03-01`) are safe features.

In [7]:
data_limits_query = """
SELECT 
    client_hash_id,
    MIN(report_date) AS client_min_date,
    MAX(report_date) AS client_max_date,
    COUNT(DISTINCT report_date) AS active_days_count,
    AVG(CASE WHEN ga4_data_available THEN 1.0 ELSE 0.0 END) AS ga4_availability_pct,
    AVG(CASE WHEN gsc_data_available THEN 1.0 ELSE 0.0 END) AS gsc_availability_pct
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
GROUP BY client_hash_id
ORDER BY ga4_availability_pct ASC
LIMIT 10;
"""
print("Sample of Client Data Limits & Structural GA4 Tracking Unavailability:")
print(con.execute(data_limits_query).df())

Sample of Client Data Limits & Structural GA4 Tracking Unavailability:
  client_hash_id client_start_date client_end_date  active_days_count  ga4_availability_pct  gsc_availability_pct
0      client_04        2026-03-01      2026-03-31                 31                   0.0                   1.0
1      client_05        2026-03-01      2026-03-31                 31                   1.0                   1.0
2      client_09        2026-03-01      2026-03-31                 31                   0.0                   1.0
3      client_10        2026-03-01      2026-03-31                 31                   1.0                   1.0
4      client_11        2026-03-01      2026-03-31                 31                   1.0                   1.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.